# Phase 1: Value Prediction — Offline TD(λ)

**Goal**: Validate forced-stay transition dynamics and TD learning in isolation.

Fixed deterministic policy (greedy toward apple), offline λ-return algorithm
(Sutton & Barto Ch. 12.1, Eq. 12.4). Ground truth from deterministic rollout.

**Success criterion**: <1% relative error for all agents on test states.

In [1]:
# =============================================================================
# CONFIG
# =============================================================================

# Model
MODEL_TYPE = "cnn"          # "mlp" | "cnn"
MLP_LAYERS = [64, 64]
CNN_LAYERS = [(16, 3), (16, 3)]
CNN_MLP_HEAD_LAYERS = [64, 64]
TRAIN_MODE = "supervised"    # "td_lambda" | "supervised"

# Environment
HEIGHT = 6
WIDTH = 6
NUM_AGENTS = 2
NUM_APPLES = 1
R_PICKER = -1.0

GAMMA = 0.99

# TD(lambda)
LAMBDA = 0.6

# Training
LR = 0.001
LR_SCHEDULE = "halve"       # "halve" | "none"
LR_HALVE_INTERVAL = 1000  # in episodes
LR_MIN = 1e-5
TRAIN_EPISODES = 10000
EVAL_FREQ = 1000            # evaluate every N episodes
STEP_LIMIT = 1000           # max steps per episode. For supervised, we cannot have any truncation
# or it will be wrong mathematicalily

# Init mode
INIT_MODE = "random"         # "fixed" | "random"
N_TEST = 100                # test trajectories (random mode only)

# Output
SEED = 42
OUTPUT_PATH = "outputs"
TAG = "cnn"


In [2]:
R_OTHER = 2.0 / (NUM_AGENTS - 1)

In [3]:
# =============================================================================
# IMPORTS
# =============================================================================
import gc
import os
import ast
import json
import time

os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
import numpy as np
import pandas as pd

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf

from helpers_forcedstay import (
    StateForcedStay,
    init_apple_positions,
    init_agent_positions,
    encode_state,
    create_model,
    env_step,
    rollout_deterministic,
    compute_trajectory_values,
    compute_lambda_returns,
    evaluate_on_states,
    greedy_toward_apple_policy,
    get_memory_mb,
    get_weight_stats,
    update_lr,
)

# Set seeds
rng = np.random.default_rng(SEED)
tf.random.set_seed(SEED)

# Papermill safety: parse string params
if isinstance(MLP_LAYERS, str):
    MLP_LAYERS = ast.literal_eval(MLP_LAYERS)
if isinstance(CNN_LAYERS, str):
    CNN_LAYERS = ast.literal_eval(CNN_LAYERS)
if isinstance(CNN_MLP_HEAD_LAYERS, str):
    CNN_MLP_HEAD_LAYERS = ast.literal_eval(CNN_MLP_HEAD_LAYERS)

INPUT_SHAPE = (HEIGHT, WIDTH, 4)
OUTPUT_DIM = 1  # V(s) is a scalar per agent
policy = greedy_toward_apple_policy

print(f"Model: {MODEL_TYPE}")
print(f"Grid: {HEIGHT}x{WIDTH}, Agents: {NUM_AGENTS}, Apples: {NUM_APPLES}")
print(f"Rewards: picker={R_PICKER}, other={R_OTHER}, gamma={GAMMA}")
print(f"TD(lambda): lambda={LAMBDA}")
print(f"Training: {TRAIN_EPISODES} episodes, eval every {EVAL_FREQ}")
print(f"LR: {LR}, schedule={LR_SCHEDULE}, halve_interval={LR_HALVE_INTERVAL}, min={LR_MIN}")
print(f"Init mode: {INIT_MODE}")
print(f"MLP: {MLP_LAYERS}, CNN: {CNN_LAYERS}, CNN head: {CNN_MLP_HEAD_LAYERS}")

start_time = time.time()
print(f"Run started: {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(start_time))}")


Model: cnn
Grid: 6x6, Agents: 2, Apples: 1
Rewards: picker=-1.0, other=2.0, gamma=0.99
TD(lambda): lambda=0.6
Training: 10000 episodes, eval every 1000
LR: 0.001, schedule=halve, halve_interval=1000, min=1e-05
Init mode: random
MLP: [64, 64], CNN: [(16, 3), (16, 3)], CNN head: [64, 64]
Run started: 2026-02-12 17:06:03


In [4]:
# =============================================================================
# INITIAL STATE + TEST SET + GROUND TRUTH
# =============================================================================

if INIT_MODE == "fixed":
    # ----- Manual state design -----
    # Agents at opposite corners, apple off-center.
    # Agent 0 is closer => picks the apple. Easy to verify by hand.
    init_state = StateForcedStay(
        agent_positions={0: (0, 0), 1: (HEIGHT - 1, WIDTH - 1)},
        apple_positions={(HEIGHT // 3, WIDTH // 3)},
        actor=0,
        H=HEIGHT, W=WIDTH,
    )

    # Roll out the deterministic trajectory (same every episode)
    train_states, train_rewards = rollout_deterministic(
        init_state, policy, R_PICKER, R_OTHER, NUM_AGENTS, STEP_LIMIT
    )
    train_T = len(train_rewards)

    # Pre-compute encodings (trajectory never changes)
    train_encodings = {
        i: np.array([encode_state(s, i) for s in train_states])
        for i in range(NUM_AGENTS)
    }
    train_agent_rewards = {
        i: [r[i] for r in train_rewards]
        for i in range(NUM_AGENTS)
    }

    # Test set = trajectory states (excluding terminal)
    test_states = train_states[:-1]
    test_true_values_dict = compute_trajectory_values(train_rewards, NUM_AGENTS, GAMMA)
    test_true_values = {i: test_true_values_dict[i] for i in range(NUM_AGENTS)}

    # Print trajectory for manual verification
    print(f"Fixed trajectory: {train_T} steps")
    for t in range(train_T):
        s = train_states[t]
        r = train_rewards[t]
        nonzero = {k: v for k, v in r.items() if v != 0}
        r_str = f", rewards={nonzero}" if nonzero else ""
        print(f"  t={t}: actor={s.actor}, agents={dict(s.agent_positions)}, "
              f"apples={s.apple_positions}{r_str}")
    s_T = train_states[-1]
    print(f"  t={train_T}: {'TERMINAL' if s_T.is_terminal() else 'STEP LIMIT'}, "
          f"agents={dict(s_T.agent_positions)}, apples={s_T.apple_positions}")

    print(f"\nGround truth values (gamma={GAMMA}):")
    for i in range(NUM_AGENTS):
        vals = test_true_values[i]
        print(f"  Agent {i}: V(s_0)={vals[0]:.6f}, V(s_{{T-1}})={vals[-1]:.6f}")

elif INIT_MODE == "random":
    test_rng = np.random.default_rng(SEED + 2000)
    test_states = []
    test_true_values = {i: [] for i in range(NUM_AGENTS)}

    for traj_idx in range(N_TEST):
        apple_pos = init_apple_positions(HEIGHT, WIDTH, NUM_APPLES, test_rng)
        agent_pos = init_agent_positions(NUM_AGENTS, HEIGHT, WIDTH, test_rng, exclude=None)
        s0 = StateForcedStay(
            agent_positions=agent_pos,
            apple_positions=apple_pos,
            actor=0, H=HEIGHT, W=WIDTH,
        )
        traj_states, traj_rewards = rollout_deterministic(
            s0, policy, R_PICKER, R_OTHER, NUM_AGENTS, STEP_LIMIT
        )
        traj_vals = compute_trajectory_values(traj_rewards, NUM_AGENTS, GAMMA)
        test_states.extend(traj_states[:-1])  # exclude terminal
        for i in range(NUM_AGENTS):
            test_true_values[i].extend(traj_vals[i])

    print(f"Random test set: {N_TEST} trajectories, {len(test_states)} total states")
    for i in range(NUM_AGENTS):
        vals = test_true_values[i]
        print(f"  Agent {i}: mean_V={np.mean(vals):.4f}, std={np.std(vals):.4f}, "
              f"min={np.min(vals):.4f}, max={np.max(vals):.4f}")

else:
    raise ValueError(f"Unknown INIT_MODE: {INIT_MODE}")


Random test set: 100 trajectories, 656 total states
  Agent 0: mean_V=0.3741, std=1.4445, min=-1.0000, max=2.0000
  Agent 1: mean_V=0.5918, std=1.4464, min=-1.0000, max=2.0000


In [5]:
# =============================================================================
# CREATE MODELS (one per agent)
# =============================================================================

models = []
for i in range(NUM_AGENTS):
    m = create_model(INPUT_SHAPE, MODEL_TYPE, MLP_LAYERS, CNN_LAYERS,
                     CNN_MLP_HEAD_LAYERS, LR, output_dim=OUTPUT_DIM)
    models.append(m)
    print(f"\n--- Agent {i} Model ---")
    m.summary()



--- Agent 0 Model ---


/u/taddmao/code/orchard-action-market/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-02-12 17:06:03.662909: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 6, 6, 16)       │           592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 6, 6, 16)       │         2,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 576)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 44,065 (172.13 KB)

...

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 6, 6, 16)       │           592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 6, 6, 16)       │         2,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 576)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 44,065 (172.13 KB)

 Trainable params: 44,065 (172.13 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:

# =============================================================================
# TRAINING LOOP
# =============================================================================

os.makedirs(OUTPUT_PATH, exist_ok=True)
csv_path = f"{OUTPUT_PATH}/phase1_{TAG}_grid{HEIGHT}.csv"

# Metadata
metadata = {
    "phase": 1,
    "task": "value_prediction_offline_td_lambda",
    "MODEL_TYPE": MODEL_TYPE,
    "MLP_LAYERS": MLP_LAYERS,
    "CNN_LAYERS": CNN_LAYERS,
    "CNN_MLP_HEAD_LAYERS": CNN_MLP_HEAD_LAYERS,
    "HEIGHT": HEIGHT, "WIDTH": WIDTH,
    "NUM_AGENTS": NUM_AGENTS, "NUM_APPLES": NUM_APPLES,
    "R_PICKER": R_PICKER, "R_OTHER": R_OTHER,
    "GAMMA": GAMMA, "LAMBDA": LAMBDA,
    "LR": LR, "LR_SCHEDULE": LR_SCHEDULE,
    "LR_HALVE_INTERVAL": LR_HALVE_INTERVAL, "LR_MIN": LR_MIN,
    "TRAIN_EPISODES": TRAIN_EPISODES, "EVAL_FREQ": EVAL_FREQ,
    "STEP_LIMIT": STEP_LIMIT,
    "INIT_MODE": INIT_MODE, "N_TEST": N_TEST,
    "SEED": SEED, "OUTPUT_PATH": OUTPUT_PATH, "TAG": TAG,
    "TRAIN_MODE": TRAIN_MODE,
}
with open(csv_path.replace('.csv', '_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=4)

first_write = True
prev_weights = {i: [] for i in range(NUM_AGENTS)}
total_grad_steps = 0
train_rng = np.random.default_rng(SEED + 1)

print(f"Training: {TRAIN_EPISODES} episodes")
print(f"Logging to: {csv_path}")

for episode in range(TRAIN_EPISODES):
    # LR scheduling (episode-based)
    current_lr = update_lr(episode, models, LR,
                           schedule=LR_SCHEDULE,
                           interval=LR_HALVE_INTERVAL,
                           min_lr=LR_MIN)

    # --- Get trajectory ---
    if INIT_MODE == "fixed":
        T = train_T
        ep_encodings = train_encodings
        ep_agent_rewards = train_agent_rewards
        ep_states = train_states
    else:
        apple_pos = init_apple_positions(HEIGHT, WIDTH, NUM_APPLES, train_rng)
        agent_pos = init_agent_positions(NUM_AGENTS, HEIGHT, WIDTH, train_rng, exclude=None)
        ep_init = StateForcedStay(
            agent_positions=agent_pos, apple_positions=apple_pos,
            actor=0, H=HEIGHT, W=WIDTH,
        )
        ep_states, ep_rewards = rollout_deterministic(
            ep_init, policy, R_PICKER, R_OTHER, NUM_AGENTS, STEP_LIMIT
        )
        T = len(ep_rewards)
        ep_encodings = {
            i: np.array([encode_state(s, i) for s in ep_states])
            for i in range(NUM_AGENTS)
        }
        ep_agent_rewards = {
            i: [r[i] for r in ep_rewards]
            for i in range(NUM_AGENTS)
        }

    # --- TD(lambda) updates for each agent ---
    episode_losses = []

    for agent_idx in range(NUM_AGENTS):
        model = models[agent_idx]
        enc = ep_encodings[agent_idx]        # (T+1, H, W, 4)

        if TRAIN_MODE == "td_lambda":
            # Frozen value estimates for all T+1 states
            v_frozen = model(tf.constant(enc), training=False).numpy().flatten()
            if ep_states[-1].is_terminal():
                v_frozen[-1] = 0.0
            G = compute_lambda_returns(
                ep_agent_rewards[agent_idx], v_frozen.tolist(), GAMMA, LAMBDA
            )
        elif TRAIN_MODE == "supervised": # we must assume traj length long enough cause we
            # do each return so they cannot be truncated
            # True values from deterministic rollout (backward sum)
            G = [0.0] * T
            v_next = 0.0
            for t in range(T - 1, -1, -1):
                G[t] = ep_agent_rewards[agent_idx][t] + GAMMA * v_next
                v_next = G[t]

        # T sequential updates (Sutton & Barto Eq. 12.4)
        for t in range(T):
            loss = model.train_on_batch(
                enc[t:t+1], np.array([[G[t]]])
            )
            episode_losses.append(float(loss))
            total_grad_steps += 1

    # --- Evaluate ---
    if episode % EVAL_FREQ == 0:
        avg_loss = np.mean(episode_losses) if episode_losses else 0.0

        row = {
            'episode': episode,
            'total_grad_steps': total_grad_steps,
            'episode_length': T,
            'avg_train_loss': avg_loss,
        }

        # Per-agent eval
        for agent_idx in range(NUM_AGENTS):
            metrics = evaluate_on_states(
                models[agent_idx], agent_idx,
                test_states, test_true_values[agent_idx],
                encode_state
            )
            for k, v in metrics.items():
                row[f'a{agent_idx}_{k}'] = v

        # Combined metrics
        row['mae_total'] = np.mean([row[f'a{i}_mae'] for i in range(NUM_AGENTS)])
        row['pct_err_total'] = np.mean([row[f'a{i}_pct_error'] for i in range(NUM_AGENTS)])
        row['current_lr'] = current_lr
        row['memory_mb'] = get_memory_mb()

        # Weight stats per agent
        for agent_idx in range(NUM_AGENTS):
            ws, prev_weights[agent_idx] = get_weight_stats(
                models[agent_idx], prev_weights[agent_idx]
            )
            for k, v in ws.items():
                row[f'a{agent_idx}_{k}'] = v

        pd.DataFrame([row]).to_csv(
            csv_path,
            mode='w' if first_write else 'a',
            header=first_write,
            index=False,
        )
        first_write = False

        if episode % (EVAL_FREQ * 10) == 0:
            print(f"Ep {episode}: loss={avg_loss:.6f}, "
                  f"mae={row['mae_total']:.4f}, "
                  f"pct_err={row['pct_err_total']:.2f}%, "
                  f"a0_bias={row['a0_bias']:.4f}, "
                  f"a1_bias={row['a1_bias']:.4f}")

print(f"\nTraining complete!")
print(f"Results saved to: {csv_path}")


Training: 10000 episodes
Logging to: outputs/phase1_cnn_grid6.csv
Ep 0: loss=2.047146, mae=1.4214, pct_err=98.11%, a0_bias=-0.6797, a1_bias=-0.2841

Training complete!
Results saved to: outputs/phase1_cnn_grid6.csv


In [7]:
# =============================================================================
# FINAL EVALUATION
# =============================================================================

print("=" * 60)
print("FINAL TEST SET RESULTS")
print("=" * 60)

for agent_idx in range(NUM_AGENTS):
    metrics = evaluate_on_states(
        models[agent_idx], agent_idx,
        test_states, test_true_values[agent_idx],
        encode_state
    )
    print(f"\nAgent {agent_idx}:")
    print(f"  MAE:        {metrics['mae']:.6f}")
    print(f"  Bias:       {metrics['bias']:.6f}")
    print(f"  Max Error:  {metrics['max_error']:.6f}")
    print(f"  Mean Pred:  {metrics['mean_pred']:.6f}")
    print(f"  Mean True:  {metrics['mean_true']:.6f}")
    print(f"  % Error:    {metrics['pct_error']:.2f}%")

overall_mae = np.mean([
    evaluate_on_states(models[i], i, test_states, test_true_values[i], encode_state)['mae']
    for i in range(NUM_AGENTS)
])
overall_pct = np.mean([
    evaluate_on_states(models[i], i, test_states, test_true_values[i], encode_state)['pct_error']
    for i in range(NUM_AGENTS)
])

print(f"\nOverall MAE: {overall_mae:.6f}")
print(f"Overall % Error: {overall_pct:.2f}%")

if overall_pct < 1.0:
    print("*** PASS: <1% relative error ***")
else:
    print("*** FAIL: >=1% relative error ***")


FINAL TEST SET RESULTS

Agent 0:
  MAE:        0.208600
  Bias:       -0.020110
  Max Error:  2.645338
  Mean Pred:  0.353954
  Mean True:  0.374064
  % Error:    14.77%

Agent 1:
  MAE:        0.275669
  Bias:       0.008379
  Max Error:  2.321715
  Mean Pred:  0.600212
  Mean True:  0.591833
  % Error:    18.56%

Overall MAE: 0.242134
Overall % Error: 16.66%
*** FAIL: >=1% relative error ***


In [8]:
# =============================================================================
# TIMING
# =============================================================================

end_time = time.time()
total_duration = end_time - start_time

metadata["total_time_seconds"] = total_duration
metadata["total_time_readable"] = time.strftime("%H:%M:%S", time.gmtime(total_duration))

with open(csv_path.replace('.csv', '_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=4)

print(f"Total time: {metadata['total_time_readable']}")


Total time: 00:04:06
